# Baseline Interpolation

In [2]:
# ============================================
# STEP 5 — Load Dataset for Baseline
# ============================================

import pandas as pd
import numpy as np

# Dataset path
file_path = "../Data/raw data/air_weather_hourly_2019_2021 (2).csv"

# Load dataset
df = pd.read_csv(file_path)

# Convert timestamp
df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"])

# Sort data chronologically
df = df.sort_values("TIMESTAMP").reset_index(drop=True)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Unique timestamps:", df["TIMESTAMP"].nunique())

Dataset loaded successfully!
Shape: (184128, 26)
Unique timestamps: 26304


In [4]:
# ============================================================
# STEP 5.1.1 — Import Required Libraries
# ============================================================

import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
# ============================================================
# STEP 5.1.2 — Define Station Columns
# ============================================================

station_columns = [
    "SHATIN",
    "TSUEN WAN",
    "CENTRAL",
    "EASTERN",
    "KWUN TONG",
    "TUEN MUN",
    "TUNG CHUNG",
    "SHAM SHUI PO",
    "SOUTHERN",
    "YUEN LONG",
    "CENTRAL/WESTERN",
    "KWAI CHUNG",
    "TSEUNG KWAN O",
    "TAI PO",
    "MONG KOK",
    "CAUSEWAY BAY"
]

print("Number of stations:", len(station_columns))

Number of stations: 16


In [6]:
# ============================================================
# STEP 5.1.3 — Define Pollutant Columns
# ============================================================

pollutant_order = [
    "Sulphur Dioxide",
    "Respirable Suspended Particulates",
    "Fine Suspended Particulates ",
    "Nitrogen Oxides",
    "Nitrogen Dioxide",
    "Carbon Monoxide",
    "Ozone"
]

print("Number of pollutants:", len(pollutant_order))
print(pollutant_order)

Number of pollutants: 7
['Sulphur Dioxide', 'Respirable Suspended Particulates', 'Fine Suspended Particulates ', 'Nitrogen Oxides', 'Nitrogen Dioxide', 'Carbon Monoxide', 'Ozone']


In [7]:
# ============================================================
# STEP 5.1.4 — Load Cleaned Dataset
# ============================================================

file_path = "../Data/raw data/air_weather_hourly_2019_2021 (2).csv"

df = pd.read_csv(file_path)

df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"])

df = df.sort_values("TIMESTAMP").reset_index(drop=True)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Unique timestamps:", df["TIMESTAMP"].nunique())

Dataset loaded successfully!
Shape: (184128, 26)
Unique timestamps: 26304


In [8]:
# ============================================================
# STEP 5.1.5 — Recreate 20% Artificial Mask
# ============================================================

observed_mask = df[station_columns].notna().astype("int8")

np.random.seed(42)

missing_rate = 0.20

mask_array = observed_mask.to_numpy(copy=True)

random_values = np.random.random(mask_array.shape)

artificial_positions = (
    (mask_array == 1) &
    (random_values < missing_rate)
)

mask_array[artificial_positions] = 0

artificial_mask = pd.DataFrame(
    mask_array,
    index=observed_mask.index,
    columns=observed_mask.columns
)

print("Artificial masking completed!")
print("Artificially masked values:",
      artificial_positions.sum())

Artificial masking completed!
Artificially masked values: 514116


In [9]:
# ============================================================
# STEP 5.1.6 — Verify Artificial Mask
# ============================================================

original_missing = df[station_columns].isna().sum().sum()

final_missing = (artificial_mask == 0).sum().sum()

artificial_missing = final_missing - original_missing

print("Original missing values:", original_missing)
print("Artificially masked values:", artificial_missing)
print("Total missing/masked values:", final_missing)

Original missing values: 372454
Artificially masked values: 514116
Total missing/masked values: 886570


In [10]:
# ============================================================
# STEP 5.1.7 — Create Dataset with Artificially Masked Values
# ============================================================

# Make a copy of the original dataset
df_masked = df.copy()

# Find artificially masked positions
artificial_mask_positions = (artificial_mask == 0) & (observed_mask == 1)

# Set only artificially masked values to NaN
for column in station_columns:
    df_masked.loc[artificial_mask_positions[column], column] = np.nan

print("Masked dataset created successfully!")

print("Original dataset shape:", df.shape)
print("Masked dataset shape:", df_masked.shape)

Masked dataset created successfully!
Original dataset shape: (184128, 26)
Masked dataset shape: (184128, 26)


In [11]:
# ============================================================
# STEP 5.2.1 — Prepare Pollutant Data for Interpolation
# ============================================================

# Select one pollutant for testing
test_pollutant = "Fine Suspended Particulates "

# Extract that pollutant
pollutant_df = df_masked[
    df_masked["POLLUTANT"] == test_pollutant
].copy()

# Sort by time
pollutant_df = pollutant_df.sort_values("TIMESTAMP")

# Keep only timestamp + station values
pollutant_data = pollutant_df[
    ["TIMESTAMP"] + station_columns
].copy()

# Use TIMESTAMP as index
pollutant_data = pollutant_data.set_index("TIMESTAMP")

print("Pollutant:", test_pollutant)
print("Shape:", pollutant_data.shape)
print("Number of timestamps:", len(pollutant_data))
print("Number of stations:", len(station_columns))

Pollutant: Fine Suspended Particulates 
Shape: (26304, 16)
Number of timestamps: 26304
Number of stations: 16


In [12]:
# ============================================================
# STEP 5.2.2 — Apply Time-Based Linear Interpolation
# ============================================================

interpolated_pollutant_data = pollutant_data.interpolate(
    method="time",
    limit_direction="both"
)

print("Interpolation completed successfully!")
print("Shape:", interpolated_pollutant_data.shape)

Interpolation completed successfully!
Shape: (26304, 16)


In [13]:
# ============================================================
# STEP 5.2.3 — Check Remaining Missing Values
# ============================================================

remaining_nan = interpolated_pollutant_data.isna().sum().sum()

print("Remaining NaN values:", remaining_nan)

if remaining_nan == 0:
    print("All missing values were successfully interpolated!")
else:
    print("Some missing values remain after interpolation.")

Remaining NaN values: 0
All missing values were successfully interpolated!


In [14]:
# ============================================================
# STEP 5.2.4 — Prepare Evaluation Data
# ============================================================

# Original pollutant data
original_pollutant_df = df[
    df["POLLUTANT"] == test_pollutant
].copy()

original_pollutant_df = original_pollutant_df.sort_values("TIMESTAMP")

original_pollutant_data = original_pollutant_df[
    ["TIMESTAMP"] + station_columns
]

original_pollutant_data = original_pollutant_data.set_index("TIMESTAMP")


# Artificial mask for same pollutant
pollutant_mask = artificial_mask.loc[
    original_pollutant_df.index
]

# Artificially masked positions only
evaluation_positions = (
    (pollutant_mask == 0) &
    (original_pollutant_data.notna())
)

print("Evaluation points:",
      evaluation_positions.sum().sum())

Evaluation points: 0


In [15]:
# ============================================================
# STEP 5.2.4 — Check Artificially Masked Positions
# ============================================================

# Original observed values
original_observed = df[station_columns].notna()

# Positions that were artificially masked
artificial_only_positions = (
    (artificial_mask == 0) &
    (original_observed == True)
)

# Select positions belonging to our test pollutant
test_indices = pollutant_df.index

test_artificial_positions = artificial_only_positions.loc[
    test_indices, station_columns
]

print("Artificially masked PM2.5 values:",
      test_artificial_positions.sum().sum())

Artificially masked PM2.5 values: 79541


In [16]:
# ============================================================
# STEP 5.2.5 — Extract Ground Truth and Predictions
# ============================================================

# Ground-truth values from original dataset
true_values = original_pollutant_data[
    test_artificial_positions
]

# Interpolated values at the same positions
predicted_values = interpolated_pollutant_data[
    test_artificial_positions
]

# Convert to 1D arrays
y_true = true_values.to_numpy().flatten()
y_pred = predicted_values.to_numpy().flatten()

print("Ground-truth values:", len(y_true))
print("Interpolated predictions:", len(y_pred))
print("Missing predictions:", np.isnan(y_pred).sum())

Ground-truth values: 420864
Interpolated predictions: 420864
Missing predictions: 420864


In [17]:
# ============================================================
# STEP 5.2.5 — Fix Evaluation Mask Alignment
# ============================================================

# Create evaluation mask using the same TIMESTAMP index
evaluation_mask = pd.DataFrame(
    test_artificial_positions.to_numpy(),
    index=pollutant_df["TIMESTAMP"],
    columns=station_columns
)

# Make sure the index matches our pollutant data
evaluation_mask = evaluation_mask.reindex(
    interpolated_pollutant_data.index
)

# Extract only artificially masked positions
y_true_matrix = original_pollutant_data.where(evaluation_mask)
y_pred_matrix = interpolated_pollutant_data.where(evaluation_mask)

# Convert to 1D arrays and remove NaN values
y_true = y_true_matrix.to_numpy().flatten()
y_pred = y_pred_matrix.to_numpy().flatten()

valid = (~np.isnan(y_true)) & (~np.isnan(y_pred))

y_true = y_true[valid]
y_pred = y_pred[valid]

print("Evaluation points:", len(y_true))
print("Missing predictions:", np.isnan(y_pred).sum())

Evaluation points: 79541
Missing predictions: 0


In [18]:
# ============================================================
# STEP 5.2.6 — Calculate MAE and RMSE
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error

# Calculate MAE
mae = mean_absolute_error(y_true, y_pred)

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print("Interpolation Baseline Results")
print("--------------------------------")
print("Evaluation points:", len(y_true))
print("MAE:", mae)
print("RMSE:", rmse)

Interpolation Baseline Results
--------------------------------
Evaluation points: 79541
MAE: 1.9483222610846787
RMSE: 2.798465116775025


In [19]:
# ============================================================
# STEP 5.2.7 — Save Interpolation Baseline Result
# ============================================================

baseline_results = pd.DataFrame({
    "Method": ["Time-based Linear Interpolation"],
    "Pollutant": [test_pollutant.strip()],
    "Evaluation_Points": [len(y_true)],
    "MAE": [mae],
    "RMSE": [rmse]
})

print(baseline_results)

                            Method                    Pollutant  \
0  Time-based Linear Interpolation  Fine Suspended Particulates   

   Evaluation_Points       MAE      RMSE  
0              79541  1.948322  2.798465  


In [20]:
# ============================================================
# STEP 5.2.8 — Save Fixed Artificial Mask
# ============================================================

# Create folder for processed data
import os

processed_folder = "../Data/processed"
os.makedirs(processed_folder, exist_ok=True)

# Save the complete fixed artificial mask
mask_path = os.path.join(
    processed_folder,
    "fixed_artificial_mask.csv"
)

artificial_mask.to_csv(mask_path, index=False)

print("Fixed artificial mask saved successfully!")
print("File:", mask_path)
print("Mask shape:", artificial_mask.shape)

Fixed artificial mask saved successfully!
File: ../Data/processed\fixed_artificial_mask.csv
Mask shape: (184128, 16)


In [21]:
# ============================================================
# STEP 5.2.9 — Interpolation for All 7 Pollutants
# ============================================================

all_baseline_results = []

for pollutant in pollutant_order:

    # Select pollutant
    pollutant_df_temp = df_masked[
        df_masked["POLLUTANT"] == pollutant
    ].copy()

    pollutant_df_temp = pollutant_df_temp.sort_values("TIMESTAMP")

    # Masked data
    masked_data = pollutant_df_temp[
        ["TIMESTAMP"] + station_columns
    ].set_index("TIMESTAMP")

    # Original data
    original_df_temp = df[
        df["POLLUTANT"] == pollutant
    ].copy()

    original_df_temp = original_df_temp.sort_values("TIMESTAMP")

    original_data = original_df_temp[
        ["TIMESTAMP"] + station_columns
    ].set_index("TIMESTAMP")

    # Interpolation
    interpolated_data = masked_data.interpolate(
        method="time",
        limit_direction="both"
    )

    # Artificially masked positions
    eval_positions = artificial_mask_positions.loc[
        pollutant_df_temp.index,
        station_columns
    ]

    # Align mask with timestamp index
    eval_mask = pd.DataFrame(
        eval_positions.to_numpy(),
        index=pollutant_df_temp["TIMESTAMP"],
        columns=station_columns
    )

    eval_mask = eval_mask.reindex(
        interpolated_data.index
    )

    # Ground truth and prediction
    true_matrix = original_data.where(eval_mask)
    pred_matrix = interpolated_data.where(eval_mask)

    y_true_temp = true_matrix.to_numpy().flatten()
    y_pred_temp = pred_matrix.to_numpy().flatten()

    # Remove invalid values
    valid = (
        (~np.isnan(y_true_temp)) &
        (~np.isnan(y_pred_temp))
    )

    y_true_temp = y_true_temp[valid]
    y_pred_temp = y_pred_temp[valid]

    # Metrics
    pollutant_mae = mean_absolute_error(
        y_true_temp,
        y_pred_temp
    )

    pollutant_rmse = np.sqrt(
        mean_squared_error(
            y_true_temp,
            y_pred_temp
        )
    )

    all_baseline_results.append({
        "Pollutant": pollutant.strip(),
        "Evaluation_Points": len(y_true_temp),
        "MAE": pollutant_mae,
        "RMSE": pollutant_rmse
    })

    print(
        f"{pollutant.strip():40s} "
        f"| Points: {len(y_true_temp):6d} "
        f"| MAE: {pollutant_mae:.4f} "
        f"| RMSE: {pollutant_rmse:.4f}"
    )

print("\nAll 7 pollutant interpolation completed!")

Sulphur Dioxide                          | Points:  79183 | MAE: 0.5306 | RMSE: 0.9620
Respirable Suspended Particulates        | Points:  79035 | MAE: 2.6686 | RMSE: 3.9632
Fine Suspended Particulates              | Points:  79541 | MAE: 1.9483 | RMSE: 2.7985
Nitrogen Oxides                          | Points:  74220 | MAE: 14.4211 | RMSE: 26.9065
Nitrogen Dioxide                         | Points:  79098 | MAE: 5.6796 | RMSE: 8.5387
Carbon Monoxide                          | Points:  43427 | MAE: 3.6213 | RMSE: 6.0041
Ozone                                    | Points:  79612 | MAE: 5.5499 | RMSE: 8.5638

All 7 pollutant interpolation completed!


In [22]:
# ============================================================
# STEP 5.2.10 — Create Final Baseline Results Table
# ============================================================

baseline_results_all = pd.DataFrame(all_baseline_results)

print("Final Interpolation Baseline Results")
print("=====================================")

display(baseline_results_all)

Final Interpolation Baseline Results


,Pollutant,Evaluation_Points,MAE,RMSE
0,Sulphur Dioxide,79183,0.530643,0.962011
1,Respirable Suspended Particulates,79035,2.668586,3.963237
2,Fine Suspended Particulates,79541,1.948322,2.798465
3,Nitrogen Oxides,74220,14.421070,26.906494
4,Nitrogen Dioxide,79098,5.679590,8.538705
5,Carbon Monoxide,43427,3.621282,6.004124
6,Ozone,79612,5.549947,8.563796


In [23]:
# ============================================================
# STEP 5.2.11 — Save Final Baseline Results
# ============================================================

results_path = "../Data/processed/interpolation_baseline_results.csv"

baseline_results_all.to_csv(
    results_path,
    index=False
)

print("Final interpolation results saved successfully!")
print("File:", results_path)
print("Number of pollutants:", len(baseline_results_all))

Final interpolation results saved successfully!
File: ../Data/processed/interpolation_baseline_results.csv
Number of pollutants: 7
